# 读取数据

In [14]:
import  pandas as pd
REE_COLS = ['class','Ce', 'Dy', 'Er', 'Eu', 'Gd', 'Ho', 'Lu', 'Nd', 'Sm', 'Th', 'Tm', 'U', 'Yb']
# 1. 读取数据
file_path = r'G:\PythonProject\Bishe\data\raw\BiShe-total_data.CSV'
df = pd.read_csv(file_path)
df

,lat,lon,class,Ce,Dy,Er,Eu,Gd,Ho,Lu,Nd,Sm,Th,Tm,U,Yb
0,40.49,-121.51,magmatic,8.45,31.79,58.36,0.49,5.73,12.14,28.29,0.30,0.85,32.0,14.37,65.0,139.69
1,40.49,-121.51,magmatic,46.28,66.46,189.80,0.51,14.86,37.68,88.34,0.57,1.69,314.0,46.54,511.0,436.00
2,40.49,-121.51,magmatic,68.14,90.61,225.64,0.85,21.73,45.38,96.36,0.72,2.46,757.0,53.23,811.0,501.76
3,40.49,-121.51,magmatic,170.38,303.39,519.85,2.95,70.00,111.31,217.11,2.78,7.38,3497.0,119.34,2127.0,1104.03
4,40.49,-121.51,magmatic,18.54,41.55,87.39,0.27,7.40,16.25,42.14,0.14,0.81,67.0,21.60,137.0,199.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7213,-25.17,-50.08,hydrothermal,1046.00,183.00,117.00,9.00,147.00,35.00,24.00,1046.00,242.00,6098.0,20.00,2046.0,162.00
7214,-25.17,-50.08,hydrothermal,1461.00,341.00,230.00,15.00,252.00,69.00,41.00,1257.00,339.00,7620.0,39.00,2035.0,298.00
7215,-25.17,-50.08,hydrothermal,1267.00,284.00,183.00,9.00,214.00,56.00,29.00,1274.00,291.00,3615.0,30.00,1386.0,217.00
7216,-25.17,-50.08,hydrothermal,1196.00,298.00,209.00,10.00,205.00,64.00,36.00,1384.00,343.00,3559.0,35.00,950.0,267.00


In [15]:
# 移除缺失标签的样本
df_labeled = df.dropna(subset=['class']).copy()
# 处理特征缺失值 - 使用中位数填充
for col in REE_COLS:
    df_labeled[col] = df_labeled[col].fillna(df_labeled[col].median())

TypeError: Cannot perform reduction 'median' with string dtype

In [13]:
df_labeled

,lat,lon,class,Ce,Dy,Er,Eu,Gd,Ho,Lu,Nd,Sm,Th,Tm,U,Yb
0,40.49,-121.51,magmatic,8.45,31.79,58.36,0.49,5.73,12.14,28.29,0.30,0.85,32.0,14.37,65.0,139.69
1,40.49,-121.51,magmatic,46.28,66.46,189.80,0.51,14.86,37.68,88.34,0.57,1.69,314.0,46.54,511.0,436.00
2,40.49,-121.51,magmatic,68.14,90.61,225.64,0.85,21.73,45.38,96.36,0.72,2.46,757.0,53.23,811.0,501.76
3,40.49,-121.51,magmatic,170.38,303.39,519.85,2.95,70.00,111.31,217.11,2.78,7.38,3497.0,119.34,2127.0,1104.03
4,40.49,-121.51,magmatic,18.54,41.55,87.39,0.27,7.40,16.25,42.14,0.14,0.81,67.0,21.60,137.0,199.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7213,-25.17,-50.08,hydrothermal,1046.00,183.00,117.00,9.00,147.00,35.00,24.00,1046.00,242.00,6098.0,20.00,2046.0,162.00
7214,-25.17,-50.08,hydrothermal,1461.00,341.00,230.00,15.00,252.00,69.00,41.00,1257.00,339.00,7620.0,39.00,2035.0,298.00
7215,-25.17,-50.08,hydrothermal,1267.00,284.00,183.00,9.00,214.00,56.00,29.00,1274.00,291.00,3615.0,30.00,1386.0,217.00
7216,-25.17,-50.08,hydrothermal,1196.00,298.00,209.00,10.00,205.00,64.00,36.00,1384.00,343.00,3559.0,35.00,950.0,267.00


In [17]:
df_labeled['class'].value_counts()

class
detrital        3437
magmatic        2517
hydrothermal     536
metamorphic      400
Name: count, dtype: int64

# 特征构造

In [16]:
import numpy as np
"""
特征类型：
1. 原始稀土元素特征 (13个)
2. 地质学比值特征 (7个)
4. 统计特征 (6个)
总计：26个特征
"""
X = df_labeled.drop(['lat','lon'], axis=1)  # 特征
y = df_labeled['class'] # 标签

features = {}

# --- 1. 原始稀土元素特征 ---
for col in REE_COLS:
    features[col] = X[col].values

# --- 3. 地质学特征 ---
# 轻稀土元素 (LREE)
features['LREE'] = (X['Ce'] + X['Nd'] + X['Sm'] + X['Eu']).values
# 重稀土元素 (HREE)
features['HREE'] = (X['Dy'] + X['Ho'] + X['Er'] + X['Yb'] + X['Lu'] + X['Tm']).values
# LREE/HREE比值 - 轻重稀土分异程度
features['LREE_HREE_ratio'] = features['LREE'] / (features['HREE'] + 1e-6)
# Eu异常 (Eu/Eu*) - 重要的地质指示指标
features['Eu_anomaly'] = X['Eu'].values / np.sqrt(X['Sm'].values * X['Gd'].values + 1e-6)
# Nd/Yb比值 - 轻重稀土分异指标
features['Nd_Yb_ratio'] = X['Nd'].values / (X['Yb'].values + 1e-6)
# Th/U比值 - 放射性元素比值
features['Th_U_ratio'] = X['Th'].values / (X['U'].values + 1e-6)
# Gd/Yb比值 - 重稀土分异程度
features['Gd_Yb_ratio'] = X['Gd'].values / (X['Yb'].values + 1e-6)

# --- 4. 统计特征 ---
ree_data = X[REE_COLS].values
features['sum_REE'] = np.sum(ree_data, axis=1)
features['mean_REE'] = np.mean(ree_data, axis=1)
features['std_REE'] = np.std(ree_data, axis=1)
features['max_REE'] = np.max(ree_data, axis=1)
features['min_REE'] = np.min(ree_data, axis=1)
features['range_REE'] = features['max_REE'] - features['min_REE']

# 转换为DataFrame
X_features = pd.DataFrame(features)

# 处理无穷大和NaN
X_features = X_features.replace([np.inf, -np.inf], np.nan)
X_features = X_features.fillna(0)
X_features

TypeError: can only concatenate str (not "float") to str

In [4]:
from sklearn.model_selection import train_test_split

# 划分特征和标签
X = X_features
y = df_labeled['class'] # 标签
# 将数据集按7:3划分为训练数据、验证数据、测试数据
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,stratify=y)
x_valid,x_test,y_valid,y_test = train_test_split(x_test,
                                                   y_test,
                                                   test_size=0.5,
                                                   random_state=42,
                                                   stratify=y_test)

In [5]:
# 标签编码
from sklearn.preprocessing import LabelEncoder

# 1. 初始化编码器
le = LabelEncoder()

# 2. 只用训练集拟合
y_train_encoded = pd.DataFrame(le.fit_transform(y_train),index=None,columns=['label'])
y_valid_encoded = pd.DataFrame(le.transform(y_valid),index=None,columns=['label'])

# 3. 测试集只做transform（不能再fit）
y_test_encoded = pd.DataFrame(le.transform(y_test),index=None,columns=['label'])

# 查看类别对应关系
print("类别映射关系：")
for i, class_name in enumerate(le.classes_):
    print(f"{class_name} -> {i}")

类别映射关系：
detrital -> 0
hydrothermal -> 1
magmatic -> 2
metamorphic -> 3


In [6]:
# 保存数据集
x_train.to_csv( 'G:/PythonProject/Bishe/data/processed/x_train_fea_move.csv')
x_valid.to_csv( 'G:/PythonProject/Bishe/data/processed/x_valid_fea_move.csv')
x_test.to_csv( 'G:/PythonProject/Bishe/data/processed/x_test_fea_move.csv')
y_train_encoded.to_csv('G:/PythonProject/Bishe/data/processed/y_train_fea_move.csv')
y_valid_encoded.to_csv('G:/PythonProject/Bishe/data/processed/y_valid_fea_move.csv')
y_test_encoded.to_csv('G:/PythonProject/Bishe/data/processed/y_test_fea_move.csv')

In [7]:
x_train

,Ce,Dy,Er,Eu,Gd,Ho,Lu,Nd,Sm,Th,...,Eu_anomaly,Nd_Yb_ratio,Th_U_ratio,Gd_Yb_ratio,sum_REE,mean_REE,std_REE,max_REE,min_REE,range_REE
4133,7.20,76.58,145.64,0.63,17.58,30.62,69.77,1.03,2.70,87.93,...,0.091443,0.003210,0.410792,0.054795,1007.05,77.465385,93.139948,320.83,0.63,320.20
1389,29.90,53.80,92.50,0.51,13.40,19.40,34.10,1.25,2.44,101.00,...,0.089191,0.006010,1.578125,0.064423,639.70,49.207692,55.836578,208.00,0.51,207.49
5666,15.00,131.00,187.00,2.20,41.00,43.00,60.00,6.00,9.20,163.00,...,0.113276,0.017241,0.920904,0.117816,1217.40,93.646154,98.384119,348.00,2.20,345.80
6749,33.00,112.00,135.00,0.90,46.00,36.00,39.00,44.00,28.00,400.00,...,0.025078,0.207547,0.442968,0.216981,2014.90,154.992308,239.911331,903.00,0.90,902.10
1225,11.60,68.70,119.20,0.28,14.30,23.90,41.70,1.66,3.30,80.00,...,0.040760,0.006561,1.000000,0.056522,720.84,55.449231,67.350073,253.00,0.28,252.72
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3665,8.95,62.50,135.00,0.27,11.20,24.80,67.20,0.63,1.29,185.00,...,0.071033,0.001957,0.345149,0.034783,1387.74,106.749231,153.326459,536.00,0.27,535.73
5273,13.70,330.00,518.00,0.36,79.20,115.00,152.00,7.10,16.00,1010.00,...,0.010113,0.007194,0.501739,0.080243,5339.36,410.720000,574.012860,2013.00,0.36,2012.64
6310,11.80,68.50,108.00,0.13,15.60,25.20,35.10,0.64,2.45,190.00,...,0.021028,0.003282,0.492228,0.080000,1060.52,81.578462,109.441291,386.00,0.13,385.87
6519,53.80,109.25,195.96,1.99,29.99,41.19,88.14,3.13,6.06,282.32,...,0.147614,0.007602,0.999221,0.072834,1552.93,119.456154,126.918631,411.76,1.99,409.77
